# 00 · Arranque del entorno

Universidad Libre — Seccional Cali · Proyecto integrado de Buenaventura, versión 5

Prepara el proyecto para **local**, **clon de GitHub**, **Google Colab** y **Drive**.
Usa el mismo `src/config.py` que el resto del pipeline: no duplica configuración.

> **Este cuaderno se detiene ante el primer fallo.** Si la instalación, las pruebas o el
> pipeline no terminan bien, lanza una excepción en lugar de continuar y dar la impresión
> de que todo funcionó.

## 1. Repositorio

La URL se toma, en este orden, de la variable de entorno `BUENAVENTURA_REPO_URL`, de un
archivo `repo_url.txt` junto al cuaderno, o de la variable `URL_REPO` de la celda
siguiente. **Si no hay ninguna, el cuaderno no inventa una dirección**: asume que el
proyecto ya está presente y lo verifica.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False


def url_repositorio() -> str:
    """Resuelve la URL sin obligar a editar el código. Devuelve cadena vacía si no existe."""
    if os.environ.get("BUENAVENTURA_REPO_URL"):
        return os.environ["BUENAVENTURA_REPO_URL"].strip()
    for base in (Path.cwd(), Path.cwd().parent):
        f = base / "repo_url.txt"
        if f.exists() and f.read_text(encoding="utf-8").strip():
            return f.read_text(encoding="utf-8").strip()
    return URL_REPO.strip()


URL_REPO = ""          # ← alternativa: pegar aquí la URL cuando el repositorio exista
CARPETA_REPO = "proyecto_integrado_buenaventura"

url = url_repositorio()
if EN_COLAB and url and not Path(CARPETA_REPO).exists():
    print(f"Clonando {url} …")
    r = subprocess.run(["git", "clone", url, CARPETA_REPO], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"El clonado falló:\n{r.stderr[-600:]}")
elif EN_COLAB and not url:
    print("Sin URL de repositorio configurada. No se inventa ninguna:")
    print("se asume que el proyecto ya está subido a la sesión.")

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError(
        "No se encontró la carpeta src/. Configure BUENAVENTURA_REPO_URL, cree "
        "repo_url.txt, o suba el proyecto a la sesión antes de continuar.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print(f"Entorno          : {'Google Colab' if EN_COLAB else 'local'}")
print(f"Raíz del proyecto: {RAIZ}")
print(f"URL configurada  : {url if url else '(ninguna, y no se inventa)'}")

## 2. Dependencias

Si la instalación falla, el cuaderno se detiene aquí.

In [ ]:
REQUISITOS = RAIZ / "requirements.txt"
if not REQUISITOS.exists():
    raise FileNotFoundError(f"Falta {REQUISITOS}")

if EN_COLAB:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUISITOS)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"La instalación de dependencias falló:\n{r.stderr[-800:]}")
    print("dependencias instaladas")

faltantes = []
for m in ("pandas", "numpy", "matplotlib", "sklearn"):
    try:
        __import__(m)
    except ImportError:
        faltantes.append(m)
if faltantes:
    raise ImportError(f"Faltan dependencias tras la instalación: {faltantes}")
print("dependencias verificadas:", "pandas, numpy, matplotlib, scikit-learn")

## 3. Datos persistentes en Drive (opcional)

Sin esto, lo que el pipeline escriba se pierde al cerrar la sesión de Colab.
`montar_drive()` reapunta las capas de datos a Drive sin introducir un segundo
sistema de rutas.

Antes de poner `USAR_DRIVE = True`, en «Mi unidad» debe existir la carpeta indicada
en `CARPETA_DRIVE` y, dentro de ella, una subcarpeta `data` con las capas `raw`,
`landing`, `trusted` y `surface`. El destino final será
`/content/drive/MyDrive/<CARPETA_DRIVE>/data`.


In [ ]:
from src import config

USAR_DRIVE = False                        # ← poner en True para persistir en Drive
CARPETA_DRIVE = "v2_buenaventura_datos"   # ← carpeta dentro de «Mi unidad»

if USAR_DRIVE:
    destino = config.montar_drive(CARPETA_DRIVE)
    if EN_COLAB and destino is None:
        raise RuntimeError("Se pidió usar Drive pero el montaje no devolvió una ruta.")

# Se crean siempre: Drive no conserva carpetas vacías, así que «landing» y
# «raw/maritimo» no llegan en la subida y hay que reponerlas antes de ejecutar.
config.asegurar_directorios()

print(f"raíz de datos : {config.DATA}")
print(f"capa raw      : {config.RAW}")
print(f"capa surface  : {config.SURFACE}")

## 4. Prueba mínima

Comprueba que el proyecto carga y que los datos están donde se esperan.

In [ ]:
from src import integracion, puertos
from src.comun import trazabilidad

d = puertos.cargar()
sp = puertos.serie_mensual(d)
cont = puertos.continuidad(sp)
cat = trazabilidad.cargar_catalogo()

print(f"serie portuaria           : {len(sp)} meses "
      f"({sp.mes.min():%Y-%m} a {sp.mes.max():%Y-%m})")
print(f"continuidad               : {cont['continua']}")
print(f"catálogo de preguntas     : {len(cat)}")
print(f"relaciones entre dominios : {len(integracion.matriz_relaciones())}")

fallos = []
if len(cat) != 52:
    fallos.append(f"el catálogo tiene {len(cat)} preguntas y deben ser 52")
if not cont["continua"]:
    fallos.append(f"la serie portuaria tiene huecos: {cont['meses_faltantes']}")
if len(sp) < 36:
    fallos.append(f"la serie portuaria solo tiene {len(sp)} meses")
if fallos:
    raise AssertionError("La prueba mínima no pasó: " + "; ".join(fallos))
print("\nPRUEBA MÍNIMA SUPERADA")

## 5. Pruebas automatizadas

Si alguna falla, el cuaderno se detiene.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", str(RAIZ / "tests"), "-q"],
                   capture_output=True, text=True, cwd=RAIZ)
print(r.stdout[-900:] or r.stderr[-900:])
if r.returncode != 0:
    raise RuntimeError(
        f"pytest falló con código {r.returncode}. No continúe hasta corregirlo.")
print("\nPRUEBAS EN VERDE")

## 6. Pipeline completo

Genera la evidencia de las 52 preguntas en `data/surface` y las figuras en
`reports/figures`. Si falla, el cuaderno se detiene.

In [ ]:
r = subprocess.run([sys.executable, "-m", "src.correr_integrado"],
                   capture_output=True, text=True, cwd=RAIZ)
print(r.stdout[-1500:] or r.stderr[-1500:])
if r.returncode != 0:
    raise RuntimeError(
        f"El pipeline falló con código {r.returncode}:\n{r.stderr[-800:]}")

import pandas as pd

traza = pd.read_csv(config.SURFACE / "matriz_trazabilidad_eda.csv")
pendientes = int((traza["estado"] == "pendiente").sum())
if len(traza) != 52 or pendientes:
    raise AssertionError(
        f"Trazabilidad incompleta: {len(traza)} preguntas, {pendientes} sin evidencia.")
print(f"\nPIPELINE COMPLETO · {len(traza)} preguntas · 0 sin evidencia")

---

Entorno listo. El EDA detallado está en `EDA_52_Preguntas_Buenaventura_V5.ipynb` y el
tablero se levanta con `streamlit run dashboard/app.py`.